# LLM Baseline — Real-Time Object Detection on KITTI

This notebook evaluates **LLM-based** visual object detection on the KITTI dataset as a baseline comparison against YOLO26n + BNN.

| Section | Model |
|---------|-------|
| A | LLaVA-1.5-7B (local, requires GPU) |
| B | Gemini 2.0 Flash (API) |
| C | Gemini 2.5 Flash (API) |
| D | ChatGPT GPT-4o (API) |
| E | Demo — inline visualization |
| F | Results comparison |

---
## 0. Setup
Run these cells once at the start of every Colab session.

In [ ]:
# 0-a. Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠ No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# 0-b. Mount Google Drive (caches HuggingFace weights between sessions)
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

In [ ]:
# 0-c. Install dependencies
!pip install -q transformers accelerate bitsandbytes pillow
!pip install -q google-genai
!pip install -q openai

In [ ]:
# 0-d. Clone / update repo
import os

REPO = 'Introduction-to-Artificial-Intelligence-Final-Project'
BRANCH = 'claude/musing-brahmagupta-7d497e'

if not os.path.exists(f'/content/{REPO}'):
    !git clone https://github.com/Appledog3572/{REPO}.git /content/{REPO}

%cd /content/{REPO}
!git checkout {BRANCH}
!git pull

import sys
sys.path.insert(0, 'llava_baseline')

---
## 1. API Keys
Fill in your keys here. Leave empty to skip the corresponding model.

In [ ]:
# === Fill in your API keys ===
GEMINI_API_KEY  = ''   # google.ai/app → Get API key
OPENAI_API_KEY  = ''   # platform.openai.com → API keys

# HuggingFace token (speeds up LLaVA weight download)
HF_TOKEN = ''

# -----------------------------------------------
import os
if HF_TOKEN:       os.environ['HF_TOKEN']       = HF_TOKEN
if GEMINI_API_KEY: os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
if OPENAI_API_KEY: os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
print('Keys configured.')

---
## 2. Shared helpers

In [ ]:
import json, time
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display as ipy_display

from dataset import KITTIDataset
from infer import InferenceRunner

# Fixed demo image IDs (val split) — mix of Cars, Pedestrians, Cyclists
DEMO_IDS = ['000001', '000013', '000017', '000035', '000000']

GT_COLOR   = (0, 200, 0)    # green
PRED_COLOR = (220, 30, 30)  # red


def _draw_boxes(image: Image.Image, gt_list: list, pred_list: list) -> Image.Image:
    """Draw GT (green) and prediction (red) boxes on a copy of the image."""
    img = image.copy()
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype('arial.ttf', 13)
    except Exception:
        font = ImageFont.load_default()

    for item, color, prefix in [(gt_list, GT_COLOR, 'GT'), (pred_list, PRED_COLOR, 'PR')]:
        for obj in item:
            x1, y1, x2, y2 = obj['bbox']
            draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
            label = f"{prefix}:{obj['class']}"
            bb = draw.textbbox((x1, y1), label, font=font)
            tw, th = bb[2]-bb[0], bb[3]-bb[1]
            ty = y1 - th - 3 if y1 - th - 3 >= 0 else y1 + 2
            draw.rectangle([x1, ty, x1+tw+4, ty+th+4], fill=color)
            draw.text((x1+2, ty+2), label, fill=(255,255,255), font=font)
    return img


def run_demo(runner: InferenceRunner, title: str, split: str = 'val'):
    """Run inference on DEMO_IDS and display results inline."""
    dataset = KITTIDataset(f'datasets/kitti_dataset', split=split)
    id_to_item = {item['image_path'].stem: item for item in dataset}

    fig, axes = plt.subplots(len(DEMO_IDS), 1, figsize=(16, 4 * len(DEMO_IDS)))
    fig.suptitle(title, fontsize=15, fontweight='bold')

    for ax, img_id in zip(axes, DEMO_IDS):
        item = id_to_item.get(img_id)
        if item is None:
            ax.set_title(f'{img_id} — not found')
            ax.axis('off')
            continue

        result = runner.run(item['image'])

        gt_list   = [{'class': g['class_name'], 'bbox': g['bbox']} for g in item['gt']]
        pred_list = [{'class': d.class_name,    'bbox': d.bbox}    for d in result.detections]

        vis = _draw_boxes(item['image'], gt_list, pred_list)
        ax.imshow(vis)
        ax.set_title(
            f"{img_id}  |  GT: {len(gt_list)}  PR: {len(pred_list)}  "
            f"latency: {result.latency_ms:.0f} ms",
            fontsize=11
        )
        ax.axis('off')

    green_patch = mpatches.Patch(color=(0,200/255,0), label='Ground Truth')
    red_patch   = mpatches.Patch(color=(220/255,30/255,30/255), label='Prediction')
    fig.legend(handles=[green_patch, red_patch], loc='lower right', fontsize=11)
    plt.tight_layout()
    plt.show()


def print_summary(results: dict):
    print(f"  mAP@0.5:       {results['mAP']:.4f}")
    for cls, ap in results['AP_per_class'].items():
        print(f"    {cls:<15s} AP={ap:.4f}")
    print(f"  Mean latency:  {results['mean_latency_ms']:.1f} ms")
    print(f"  Mean FPS:      {results['mean_FPS']:.2f}")
    print(f"  Model size:    {results['model_size_MB']:.1f} MB")

print('Helpers loaded.')

---
# A. LLaVA-1.5-7B
Local model. Requires T4 GPU and ~8 GB VRAM (loaded with 4-bit quantization).

In [ ]:
# A.1 Full evaluation
!mkdir -p results
!python llava_baseline/evaluate.py \
    --split val \
    --mode llava \
    --max-images 100 \
    --output-json results/llava_val.json

In [ ]:
# A.2 Print summary
llava_results = json.loads(Path('results/llava_val.json').read_text())
print('=== LLaVA-1.5-7B ===')
print_summary(llava_results)

In [ ]:
# A.3 Demo — 5 fixed images, inline display
llava_runner = InferenceRunner(mode='llava')
run_demo(llava_runner, 'LLaVA-1.5-7B  |  Green = GT   Red = Prediction')

---
# B. Gemini 2.0 Flash
API-based. Fill in `GEMINI_API_KEY` in Section 1.

In [ ]:
# B.1 Full evaluation
!mkdir -p results
!python llava_baseline/evaluate.py \
    --split val \
    --mode gemini \
    --model-id gemini-2.0-flash \
    --max-images 100 \
    --api-key "$GEMINI_API_KEY" \
    --output-json results/gemini20_val.json

In [ ]:
# B.2 Print summary
gemini20_results = json.loads(Path('results/gemini20_val.json').read_text())
print('=== Gemini 2.0 Flash ===')
print_summary(gemini20_results)

In [ ]:
# B.3 Demo
gemini20_runner = InferenceRunner(mode='gemini', model_id='gemini-2.0-flash', api_key=GEMINI_API_KEY)
run_demo(gemini20_runner, 'Gemini 2.0 Flash  |  Green = GT   Red = Prediction')

---
# C. Gemini 2.5 Flash
API-based. Same key as Gemini 2.0.

In [ ]:
# C.1 Full evaluation
!mkdir -p results
!python llava_baseline/evaluate.py \
    --split val \
    --mode gemini \
    --model-id gemini-2.5-flash \
    --max-images 100 \
    --api-key "$GEMINI_API_KEY" \
    --output-json results/gemini25_val.json

In [ ]:
# C.2 Print summary
gemini25_results = json.loads(Path('results/gemini25_val.json').read_text())
print('=== Gemini 2.5 Flash ===')
print_summary(gemini25_results)

In [ ]:
# C.3 Demo
gemini25_runner = InferenceRunner(mode='gemini', model_id='gemini-2.5-flash', api_key=GEMINI_API_KEY)
run_demo(gemini25_runner, 'Gemini 2.5 Flash  |  Green = GT   Red = Prediction')

---
# D. ChatGPT GPT-4o
API-based. Fill in `OPENAI_API_KEY` in Section 1.

In [ ]:
# D.1 Full evaluation
!mkdir -p results
!python llava_baseline/evaluate.py \
    --split val \
    --mode gpt4o \
    --model-id gpt-4o \
    --max-images 100 \
    --api-key "$OPENAI_API_KEY" \
    --output-json results/gpt4o_val.json

In [ ]:
# D.2 Print summary
gpt4o_results = json.loads(Path('results/gpt4o_val.json').read_text())
print('=== GPT-4o ===')
print_summary(gpt4o_results)

In [ ]:
# D.3 Demo
gpt4o_runner = InferenceRunner(mode='gpt4o', model_id='gpt-4o', api_key=OPENAI_API_KEY)
run_demo(gpt4o_runner, 'GPT-4o  |  Green = GT   Red = Prediction')

---
# F. Results Comparison
Aggregates all available results into a single table.

In [ ]:
import json
from pathlib import Path

result_files = {
    'LLaVA-1.5-7B':      'results/llava_val.json',
    'Gemini 2.0 Flash':  'results/gemini20_val.json',
    'Gemini 2.5 Flash':  'results/gemini25_val.json',
    'GPT-4o':            'results/gpt4o_val.json',
}

rows = []
for name, path in result_files.items():
    if not Path(path).exists():
        continue
    r = json.loads(Path(path).read_text())
    rows.append({
        'Model':       name,
        'mAP@0.5':     r['mAP'],
        'AP Car':      r['AP_per_class']['Car'],
        'AP Ped':      r['AP_per_class']['Pedestrian'],
        'AP Cyclist':  r['AP_per_class']['Cyclist'],
        'FPS':         r['mean_FPS'],
        'Latency(ms)': r['mean_latency_ms'],
        'Size(MB)':    r['model_size_MB'],
        'N':           r['n_images'],
    })

# Print table
if rows:
    header = list(rows[0].keys())
    col_w  = [max(len(h), max(len(str(r[h])) for r in rows)) + 2 for h in header]
    sep    = '+' + '+'.join('-' * w for w in col_w) + '+'
    fmt    = '|' + '|'.join(f'{{:<{w}}}' for w in col_w) + '|'
    print(sep)
    print(fmt.format(*header))
    print(sep)
    for r in rows:
        print(fmt.format(*[str(r[h]) for h in header]))
    print(sep)
else:
    print('No result files found. Run the evaluation cells first.')

---
## Save results to Google Drive

In [ ]:
import shutil, os
dest = '/content/drive/MyDrive/llm_baseline_results'
os.makedirs(dest, exist_ok=True)
for f in Path('results').glob('*.json'):
    shutil.copy(f, dest)
print(f'Saved to {dest}')